# 01, the first backlog chart

The first chart of Q1 divided the CVEs the NVD analyzed within 30 days by every CVE published that month. Rejected CVEs were in the denominator, and the NVD never analyzes a rejected CVE, so every rejection read as a CVE the NVD skipped. This notebook shows how much that moved the line, and why the registered Q1 leaves rejected CVEs out.

In [1]:
import duckdb
import polars as pl

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_width_chars(110)
con = duckdb.connect("data/lake/unscheduled.duckdb", read_only=True)
con.execute("set TimeZone = 'UTC'").fetchall()
cut = con.execute("select data_cut from config.lake_build").fetchall()[0][0]
print("lake cut at", cut)

lake cut at 2026-11-13 06:00:00+00:00


In [2]:
first = con.sql('''
    select published_month as month,
           count(*) as cves,
           count(*) filter (where state = 'REJECTED') as rejected,
           avg(case when first_analysis_at is not null
                     and first_analysis_at <= published + interval 30 day then 1 else 0 end) as share_first
    from gold.cve_current
    where published >= timestamptz '2020-01-01 00:00:00+00'
      and published + interval 30 day <= $cut
    group by all order by month
''', params={"cut": cut}).pl()
first.tail(6)

shape: (6, 4)
┌────────────┬───────┬──────────┬─────────────┐
│ month      ┆ cves  ┆ rejected ┆ share_first │
│ ---        ┆ ---   ┆ ---      ┆ ---         │
│ date       ┆ i64   ┆ i64      ┆ f64         │
╞════════════╪═══════╪══════════╪═════════════╡
│ 2026-05-01 ┆ 6982  ┆ 49       ┆ 0.539387    │
│ 2026-06-01 ┆ 8008  ┆ 59       ┆ 0.531094    │
│ 2026-07-01 ┆ 9908  ┆ 149      ┆ 0.495458    │
│ 2026-08-01 ┆ 12721 ┆ 464      ┆ 0.297304    │
│ 2026-09-01 ┆ 15075 ┆ 146      ┆ 0.308988    │
│ 2026-10-01 ┆ 7278  ┆ 60       ┆ 0.357378    │
└────────────┴───────┴──────────┴─────────────┘

The fix: rejected CVEs out of the population, as the registered plan states it, and a CVE judged at 30 days only once 30 days have passed (the censoring the mart applies).

In [3]:
fixed = con.sql('''
    select published_month as month, avg(cast(analyzed_30d as integer)) as share_fixed
    from gold.mart_backlog_q1
    group by all order by month
''').pl()
both = first.join(fixed, on="month").with_columns(
    (pl.col("share_fixed") - pl.col("share_first")).alias("moved_by")
)
both.select("month", "rejected", "share_first", "share_fixed", "moved_by").tail(6)

shape: (6, 5)
┌────────────┬──────────┬─────────────┬─────────────┬──────────┐
│ month      ┆ rejected ┆ share_first ┆ share_fixed ┆ moved_by │
│ ---        ┆ ---      ┆ ---         ┆ ---         ┆ ---      │
│ date       ┆ i64      ┆ f64         ┆ f64         ┆ f64      │
╞════════════╪══════════╪═════════════╪═════════════╪══════════╡
│ 2026-05-01 ┆ 49       ┆ 0.539387    ┆ 0.541757    ┆ 0.00237  │
│ 2026-06-01 ┆ 59       ┆ 0.531094    ┆ 0.534407    ┆ 0.003313 │
│ 2026-07-01 ┆ 149      ┆ 0.495458    ┆ 0.502101    ┆ 0.006642 │
│ 2026-08-01 ┆ 464      ┆ 0.297304    ┆ 0.307498    ┆ 0.010194 │
│ 2026-09-01 ┆ 146      ┆ 0.308988    ┆ 0.31201     ┆ 0.003022 │
│ 2026-10-01 ┆ 60       ┆ 0.357378    ┆ 0.360211    ┆ 0.002832 │
└────────────┴──────────┴─────────────┴─────────────┴──────────┘

In [4]:
worst = both.sort("moved_by", descending=True).head(3)
print("months the rejected CVEs pulled down most:")
for row in worst.iter_rows(named=True):
    print(f"  {row['month']}: {row['rejected']} rejected, {row['share_first']:.1%} first, {row['share_fixed']:.1%} fixed")

months the rejected CVEs pulled down most:
  2023-09-01: 680 rejected, 76.2% first, 100.0% fixed
  2021-01-01: 427 rejected, 78.2% first, 99.9% fixed
  2021-12-01: 465 rejected, 80.9% first, 99.8% fixed


What changed in the build: dbt/models/marts/mart_backlog_q1.sql keeps state <> 'REJECTED' in its population, and the Q1 departure note in results/backlog/q1.json says why.